# Vail / Eagle River (VCRC2) Runoff Analysis

Two data sources:
- **Cell 1** — NOAA CBRFC Ensemble Streamflow Prediction (ESP) forecast for VCRC2
- **Cell 2** — SNODAS snowpack statistics for the VCRC2H basin

In [ ]:
# ── Cell 1: NOAA CBRFC ESP Forecast ──────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import io

NOAA_URL = (
    "https://www.cbrfc.noaa.gov/wsup/graph/esptxt.py"
    "?id=VCRC2&year=2026&qpf=0&db=&csv=1"
)

resp = requests.get(NOAA_URL, timeout=30)
resp.raise_for_status()
raw = resp.text

# ── Parse ─────────────────────────────────────────────────────────────────────
# The CBRFC ESP CSV contains comment / metadata lines before the data block.
# Find the first line that starts with a digit or looks like a header row.
lines = raw.splitlines()

# Identify header line (contains commas and a date-like or column token)
header_idx = None
for i, line in enumerate(lines):
    if line.strip() and "," in line:
        header_idx = i
        break

if header_idx is None:
    raise ValueError("Could not locate CSV header in NOAA response.")

csv_text = "\n".join(lines[header_idx:])
df_noaa = pd.read_csv(io.StringIO(csv_text))
df_noaa.columns = df_noaa.columns.str.strip()

print("Shape:", df_noaa.shape)
print("Columns:", df_noaa.columns.tolist())
df_noaa.head(10)


In [ ]:
# ── Plot ESP Forecast ─────────────────────────────────────────────────────────
# CBRFC ESP CSV columns are typically:
#   Date (or Month/Day), 10%, 20%, 30%, 40%, 50%, 60%, 70%, 80%, 90%, Obs, Mean
# Detect date / period column automatically.

df = df_noaa.copy()

# Find the date / label column (first non-numeric-named column or 'Date')
date_col = df.columns[0]
try:
    df[date_col] = pd.to_datetime(df[date_col])
    x = df[date_col]
    x_label = "Date"
except Exception:
    x = df[date_col].astype(str)
    x_label = date_col

# Exceedance probability columns (anything that looks like "##%" or "p##")
pct_cols = [c for c in df.columns if c.strip().endswith("%") or c.strip().lower().startswith("p")]
# Fallback: all numeric columns except the date col
if not pct_cols:
    pct_cols = [c for c in df.columns if c != date_col and pd.api.types.is_numeric_dtype(df[c])]

obs_col  = next((c for c in df.columns if "obs"  in c.lower()), None)
mean_col = next((c for c in df.columns if "mean" in c.lower()), None)

fig, ax = plt.subplots(figsize=(12, 5))

# Fan chart: fill between outer percentiles, then inner
pct_sorted = sorted(pct_cols, key=lambda c: float(c.strip().rstrip("%")) if c.strip().rstrip("%").replace(".","").isdigit() else 50)
colors = plt.cm.Blues_r
n = len(pct_sorted)
for i in range(n // 2):
    lo, hi = pct_sorted[i], pct_sorted[-(i + 1)]
    if lo != hi:
        ax.fill_between(
            x, df[lo], df[hi],
            alpha=0.25 + 0.05 * i,
            color=colors(0.3 + 0.05 * i),
            label=f"{lo}–{hi}"
        )

# Median line
median_col = next((c for c in pct_sorted if "50" in c), None)
if median_col:
    ax.plot(x, df[median_col], color="steelblue", lw=2, label="50% (median)")

if mean_col:
    ax.plot(x, df[mean_col], color="darkorange", lw=1.5, ls="--", label="Mean")

if obs_col:
    ax.plot(x, df[obs_col], color="black", lw=2, ls="-", marker="o", ms=3, label="Observed")

ax.set_title("NOAA CBRFC ESP Streamflow Forecast — VCRC2 (2026)", fontsize=13, fontweight="bold")
ax.set_xlabel(x_label)
ax.set_ylabel("Volume (KAF)" if "kaf" in raw.lower() else "Flow")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
plt.show()


In [ ]:
# ── Cell 2: SNODAS Snowpack Statistics ───────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import io

SNODAS_URL = (
    "https://snodas.cdss.state.co.us/data/SnowpackStatisticsByBasin/"
    "SnowpackStatisticsByBasin_VCRC2H_F.csv"
)

resp2 = requests.get(SNODAS_URL, timeout=30)
resp2.raise_for_status()

df_snow = pd.read_csv(io.StringIO(resp2.text))
df_snow.columns = df_snow.columns.str.strip()

print("Shape:", df_snow.shape)
print("Columns:", df_snow.columns.tolist())
df_snow.head(10)


In [ ]:
# ── Plot SNODAS Snowpack ──────────────────────────────────────────────────────
# Typical SNODAS columns (Colorado DWR format):
#   Date, LocalDate, SWE_Mean_in, SnowDepth_Mean_in, SWE_Volume_acft,
#   SnowCover_pct, SWE_PctOfMedian, SWE_PctOf30YrNormal, ...

ds = df_snow.copy()

# Identify date column
date_col = next(
    (c for c in ds.columns if "date" in c.lower() or "time" in c.lower()),
    ds.columns[0]
)
ds[date_col] = pd.to_datetime(ds[date_col], errors="coerce")
ds = ds.dropna(subset=[date_col]).sort_values(date_col)

# ── Helper: find column by keyword list ─────────────────────────────────────
def find_col(df, *keywords):
    for kw in keywords:
        match = next((c for c in df.columns if kw.lower() in c.lower()), None)
        if match:
            return match
    return None

swe_col    = find_col(ds, "swe", "snow_water", "snowwater")
depth_col  = find_col(ds, "depth", "snowdepth")
cover_col  = find_col(ds, "cover", "percent_covered", "pct")
pct_med    = find_col(ds, "pctofmedian", "pct_median", "median")
vol_col    = find_col(ds, "volume", "vol_acft", "acft")

# Build subplot grid based on what columns exist
panels = []
if swe_col:   panels.append((swe_col,   "SWE (in)",           "steelblue"))
if depth_col: panels.append((depth_col, "Snow Depth (in)",    "slateblue"))
if vol_col:   panels.append((vol_col,   "SWE Volume (ac-ft)", "teal"))
if pct_med:   panels.append((pct_med,   "% of Median SWE",    "darkorange"))
if cover_col: panels.append((cover_col, "Snow Cover (%)",     "mediumseagreen"))

# Fall back to all numeric columns if nothing matched
if not panels:
    num_cols = [c for c in ds.columns if c != date_col and pd.api.types.is_numeric_dtype(ds[c])]
    panels = [(c, c, "steelblue") for c in num_cols[:5]]

n_panels = len(panels)
fig, axes = plt.subplots(n_panels, 1, figsize=(12, 3.2 * n_panels), sharex=True)
if n_panels == 1:
    axes = [axes]

for ax, (col, ylabel, color) in zip(axes, panels):
    ax.plot(ds[date_col], ds[col], color=color, lw=1.5)
    ax.fill_between(ds[date_col], ds[col], alpha=0.15, color=color)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.1f}"))
    ax.grid(True, linestyle=":", alpha=0.5)

    # Draw 100% line for pct-of-median panel
    if "median" in ylabel.lower() or "normal" in ylabel.lower():
        ax.axhline(100, color="gray", lw=1, ls="--", label="100% of median")
        ax.legend(fontsize=8)

axes[0].set_title(
    "SNODAS Snowpack Statistics — VCRC2H Basin",
    fontsize=13, fontweight="bold"
)
axes[-1].set_xlabel("Date")
fig.tight_layout()
plt.show()
